# Qiskit Introduction

This interactive notebook serves as a hands-on introduction to **Qiskit**, the open-source SDK developed by IBM for working with quantum computers at the level of circuits, pulses, and algorithms.

Throughout this document, we will explore the algebraic foundations of quantum states, the creation of unitary operators, and circuit simulation using the local Aer backend.

In [ ]:
import sys

import qiskit

# Environment and version check
print(f"Executable: {sys.executable}")
print(f"Python Version: {sys.version}")
print(f"Qiskit Version: {qiskit.__version__}")

## Statevectors and Linear Algebra

In quantum computing, the state of a qubit is mathematically represented as a two-dimensional column vector. Using `numpy`, we can define the basis states of quantum computing: $\ket{0}$ and $\ket{1}$.

In [ ]:
import numpy as np
from qiskit.visualization import array_to_latex

# Defining basis states
ket0 = np.array([[1], [0]])
ket1 = np.array([[0], [1]])

# Simple superposition
superposition = ket0 / 2 + ket1 / 2
print("Vector in superposition:")
print(superposition)

print("\nLatex representation:")
array_to_latex(superposition, prefix="\\text{Statevector: }", precision=3)

Qiskit provides the `Statevector` class to handle these states in a much more powerful and elegant way, allowing us to easily validate and visualize them with native LaTeX support.

In [ ]:
from numpy import sqrt
from qiskit.quantum_info import Statevector

# Creating different state vectors
u = Statevector([1 / sqrt(2), 1 / sqrt(2)])  # Equiprobable superposition state
v = Statevector([(1 + 2.0j) / 3, -2 / 3])  # State with complex amplitudes
w = Statevector([1 / 3, 2 / 3])  # Invalid state (not normalized)

# Visualizing state 'u' in different formats
display(u.draw("text"))
display(u.draw("latex"))

# Mathematical validation of the states (the sum of probabilities must be 1)
print(f"Is 'u' a valid state? {u.is_valid()}")
print(f"Is 'w' a valid state? {w.is_valid()}")

## Measurement and Probability

Quantum mechanics dictates that when a system in superposition is measured, it collapses to one of its basis states. We can simulate this collapse or extract the theoretical probability distribution without destroying the state.

In [ ]:
from qiskit.visualization import plot_histogram

display(v.draw("latex"))

# Simulating a single measurement
outcome, state = v.measure()
print(f"Measured outcome: {outcome}\nPost-measurement state:")
display(state.draw("latex"))

# Extracting the statistical distribution after 1000 measurements
statistics = v.sample_counts(1000)
plot_histogram(statistics)

## Quantum Operators (Logic Gates)

Transformations on quantum states are applied using logic gates, which are represented as unitary matrices. In Qiskit, we use the `Operator` class to define them and the `evolve()` method to apply them to a `Statevector`.

In [ ]:
from qiskit.quantum_info import Operator

# Matrix definition of common gates
Y = Operator([[0, -1.0j], [1.0j, 0]])
H = Operator([[1 / sqrt(2), 1 / sqrt(2)], [1 / sqrt(2), -1 / sqrt(2)]])
S = Operator([[1, 0], [0, 1.0j]])
T = Operator([[1, 0], [0, (1 + 1.0j) / sqrt(2)]])

print("T-gate matrix:")
display(T.draw("latex"))

# Evolving the initial state |0> by applying the matrices sequentially
initial_state = Statevector([1, 0])
evolved_state = initial_state.evolve(H).evolve(T).evolve(H).evolve(S).evolve(Y)

print("Final state after evolution:")
display(evolved_state.draw("latex"))

## Quantum Circuits

Although working with matrices is fundamental for the theoretical background, modern quantum programming abstracts this through the use of quantum circuits. The `QuantumCircuit` class allows us to assemble these operators visually and systematically.

In [ ]:
from qiskit import QuantumCircuit

# We can recreate the exact same mathematical evolution using a circuit
circuit = QuantumCircuit(1)  # 1-qubit circuit

circuit.h(0)
circuit.t(0)
circuit.h(0)
circuit.s(0)
circuit.y(0)

# Qiskit allows rendering the circuit using matplotlib ('mpl')
display(circuit.draw(output="mpl"))

# We can also extract the global unitary operator resulting from the entire circuit
global_matrix = Operator.from_circuit(circuit)
display(global_matrix.draw("latex"))

## Full Simulation and Execution with Aer

Finally, we will build an entangled circuit with multiple qubits and classical bits, and we will execute it using Qiskit Aer's high-performance simulators.

In [ ]:
from qiskit import transpile
from qiskit_aer import Aer

# Creating a circuit with 2 qubits and 2 classical bits
circ = QuantumCircuit(2, 2)

# Applying gates
circ.h(0)  # Hadamard gate on the first qubit (superposition)
circ.x(1)  # NOT gate on the second qubit
circ.z(1)  # Z gate on the second qubit

# Measurement: map the qubits (0,1) to the classical bits (0,1)
circ.measure((0, 1), (0, 1))

display(circ.draw("mpl"))

# Simulator configuration (Quantum Assembly Language backend)
backend = Aer.get_backend("qasm_simulator")

# Transpilation: optimizes and translates the circuit into native backend instructions
qc_transpiled = transpile(circ, backend)

# Executing the job on the simulator with 10,000,000 shots
job = backend.run(qc_transpiled, shots=10000000)

# Extracting and visualizing results
result = job.result()
counts = result.get_counts()

plot_histogram(counts)

Because the number of shots is finite, we will observe small statistical variations around the theoretical 50/50 split, a phenomenon explained by the central limit theorem.